# ShellBreaker — ResNet50 Training Notebook

**Purpose:** Fine-tune ResNet50 on opcode-matrix grayscale PNGs to detect Java memory webshells.

## Before you start
1. Runtime → Change runtime type → **T4 GPU** (or A100 if using Colab Pro)
2. Upload `output.zip` to Google Drive (see TRAINING_GUIDE.md Step 2)
3. Upload `scripts/04_train_resnet50.py` to Google Drive too
4. Run all cells top-to-bottom

**Expected time:** ~20–40 min on T4 depending on dataset size.

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── EDIT THIS PATH to match where you uploaded your files ──
DRIVE_BASE = '/content/drive/MyDrive/ShellBreaker'

import os
print('Drive files:', os.listdir(DRIVE_BASE))

## Step 2 — Install dependencies

In [ ]:
# Colab already has torch/torchvision; just install missing packages
!pip install -q scikit-learn tqdm pillow

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Step 3 — Extract dataset from Drive

In [ ]:
import shutil, os

WORK_DIR = '/content/ShellBreaker'
os.makedirs(WORK_DIR, exist_ok=True)

# Extract the output.zip you uploaded to Drive
ZIP_PATH = f'{DRIVE_BASE}/output.zip'
!unzip -q "{ZIP_PATH}" -d "{WORK_DIR}"

# Copy the training script
!cp "{DRIVE_BASE}/04_train_resnet50.py" "{WORK_DIR}/"

# Verify
!echo '=== output/ contents ===' && ls {WORK_DIR}/output/
!echo '=== PNG count ===' && find {WORK_DIR}/output -name '*.png' | wc -l
!echo '=== dataset.csv head ===' && head -5 {WORK_DIR}/output/dataset.csv

## Step 4 — (Optional) Quick sanity check — visualise a few samples

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import random

csv_path = Path(f'{WORK_DIR}/output/dataset.csv')
lines = csv_path.read_text().splitlines()[1:]   # skip header
random.shuffle(lines)
samples = [(l.split(',')[0], int(l.split(',')[1])) for l in lines[:8]]

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, (path, label) in zip(axes.flat, samples):
    img_path = path if Path(path).is_absolute() else Path(WORK_DIR) / path
    img = Image.open(img_path)
    ax.imshow(img, cmap='gray')
    ax.set_title('WEBSHELL' if label == 1 else 'benign', 
                 color='red' if label == 1 else 'green', fontsize=10)
    ax.axis('off')
plt.suptitle('Opcode adjacency matrix PNGs (149×149)', fontsize=12)
plt.tight_layout()
plt.show()

## Step 5 — Patch paths and run training

In [ ]:
# Patch the ROOT path inside the script so it points to our working dir
import re

script_path = f'{WORK_DIR}/04_train_resnet50.py'
code = open(script_path).read()

# Override ROOT to point at /content/ShellBreaker
code = re.sub(
    r'ROOT = Path\(__file__\)\.parent\.parent',
    f'ROOT = Path("{WORK_DIR}")',
    code
)
open(script_path, 'w').write(code)
print('Path patched. Starting training...')

In [ ]:
# Run training — this takes 20-40 minutes on T4
%cd {WORK_DIR}
!python 04_train_resnet50.py

## Step 6 — Inspect results

In [ ]:
import json

report = json.load(open(f'{WORK_DIR}/output/training_report.json'))

print('\n=== Per-run results (threshold-optimised) ===')
for r in report['runs']:
    t = r.get('test_opt', r['test'])
    cm = r.get('confusion_matrix_opt', r['confusion_matrix'])
    thr = r.get('opt_threshold', 0.50)
    print(f"  Run {r['run']} (thr={thr:.4f}): "
          f"acc={t['accuracy']:.4f}  pre={t['precision']:.4f}  "
          f"rec={t['recall']:.4f}  f1={t['f1']:.4f}  auc={t['auc_roc']:.4f}")
    print(f"    CM: TN={cm[0][0]} FP={cm[0][1]} | FN={cm[1][0]} TP={cm[1][1]}")

print('\n=== 3-Run Average (threshold-optimised) ===')
avg = report.get('average_opt', report['average'])
std = report.get('std_opt',     report['std'])
for k in ['accuracy', 'precision', 'recall', 'f1', 'auc_roc']:
    print(f'  {k:12s}: {avg[k]:.4f}  ±{std[k]:.4f}')
print(f"\n  Inference threshold saved: {report.get('inference_threshold', 0.70)}")

fl = report.get('fileless_generalisation', {})
if fl:
    print('\n=== Cross-type generalisation (fileless — never in training) ===')
    fl_cm = fl.get('confusion_matrix', [['?','?'],['?','?']])
    print(f"  recall   : {fl.get('recall', 0):.4f}")
    print(f"  precision: {fl.get('precision', 0):.4f}")
    print(f"  f1       : {fl.get('f1', 0):.4f}")
    print(f"  CM: TN={fl_cm[0][0]} FP={fl_cm[0][1]} | FN={fl_cm[1][0]} TP={fl_cm[1][1]}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

metrics = ['accuracy', 'precision', 'recall', 'f1', 'auc_roc']
labels  = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC']
runs    = report['runs']
avg     = report['average']
std     = report['std']
colors  = ['#4C72B0', '#DD8452', '#55A868']

# ── 1. Per-run metrics + average ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x, w = np.arange(len(metrics)), 0.25
ax = axes[0]
for i, r in enumerate(runs):
    ax.bar(x + i*w, [r['test'][m] for m in metrics], w,
           label=f'Run {r["run"]}', color=colors[i])
ax.set_xticks(x + w); ax.set_xticklabels(labels)
ax.set_ylim(0, 1.1); ax.set_ylabel('Score')
ax.set_title('Per-run Test Metrics'); ax.legend()
for s in ['top', 'right']: ax.spines[s].set_visible(False)

ax = axes[1]
avgs = [avg[m] for m in metrics]
stds = [std[m] for m in metrics]
bars = ax.bar(labels, avgs, color='#4C72B0', alpha=0.85,
              yerr=stds, capsize=6, error_kw={'elinewidth':2, 'ecolor':'#333'})
for bar, v, s in zip(bars, avgs, stds):
    ax.text(bar.get_x() + bar.get_width()/2, v + s + 0.02,
            f'{v:.3f}', ha='center', fontsize=9)
ax.set_ylim(0, 1.1); ax.set_ylabel('Score')
ax.set_title('3-Run Average ± Std')
for s in ['top', 'right']: ax.spines[s].set_visible(False)

plt.tight_layout()
plt.savefig(f'{WORK_DIR}/output/metrics_bar.png', dpi=150)
plt.show()

# ── 2. Confusion matrices: all runs + fileless ────────────────────────────────
fl = report.get('fileless_generalisation', {})
n_cols = len(runs) + (1 if fl.get('confusion_matrix') else 0)
fig, axes = plt.subplots(1, n_cols, figsize=(5*n_cols, 4))
if n_cols == 1: axes = [axes]

for i, r in enumerate(runs):
    cm = np.array(r['confusion_matrix'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['benign','webshell'],
                yticklabels=['benign','webshell'],
                ax=axes[i], cbar=False)
    axes[i].set_title(f'Run {r["run"]}  F1={r["test"]["f1"]:.4f}')
    axes[i].set_xlabel('Predicted'); axes[i].set_ylabel('True')

if fl.get('confusion_matrix'):
    fl_cm = np.array(fl['confusion_matrix'])
    sns.heatmap(fl_cm, annot=True, fmt='d', cmap='Oranges',
                xticklabels=['benign','webshell'],
                yticklabels=['benign','webshell'],
                ax=axes[-1], cbar=False)
    axes[-1].set_title(
        f'Fileless generalisation\nRecall={fl.get("recall",0):.4f}  F1={fl.get("f1",0):.4f}')
    axes[-1].set_xlabel('Predicted'); axes[-1].set_ylabel('True')

plt.tight_layout()
plt.savefig(f'{WORK_DIR}/output/confusion_matrices.png', dpi=150)
plt.show()
print('Saved: metrics_bar.png, confusion_matrices.png')

## Step 7 — Copy results back to Google Drive

In [ ]:
import shutil, os, json

SAVE_DIR = f'{DRIVE_BASE}/results'
os.makedirs(SAVE_DIR, exist_ok=True)

# training_report.json now includes per-epoch history for all runs
for fname in ['model_best.pt', 'training_report.json',
              'training_curves.png', 'confusion_matrix.png']:
    src = f'{WORK_DIR}/output/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'{SAVE_DIR}/{fname}')
        print(f'  Copied {fname} → Drive')
    else:
        print(f'  [warn] {fname} not found')

# Confirm history is present in the saved report
report_path = f'{SAVE_DIR}/training_report.json'
if os.path.exists(report_path):
    rep = json.load(open(report_path))
    has_history = all('history' in r for r in rep.get('runs', []))
    print(f'\n  Per-epoch history included in report: {has_history}')
    if has_history:
        total_epochs = sum(len(r['history']) for r in rep['runs'])
        print(f'  Total epochs recorded across all runs: {total_epochs}')

print('\nDone! Download model_best.pt from Google Drive to output/ locally.')